In [1]:
from datetime import date
import hisepy
import os
import polars as pl
import re

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

In [3]:
out_files = []

## Helper functions

In [4]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Input files stored in HISE

In [5]:
in_uuids = {
    'deg': 'a134530f-9ac4-4d37-bf01-97c73bf1ce24', # RNA: Differentially Expressed Genes
    'hallmark': '4f05f2a0-59d2-4d04-b948-70d541613e71', #RNA: Hallmark Gene Set Enrichment Analysis results
    'reactome': 'ad8d7320-399c-41ff-8b2f-b840d8eae324', #RNA: Reactome Gene Set Enrichment Analysis results
    'dap': '681e1f11-708c-4ab4-8ed0-87ef895df152', # ATAC: Differentially Accessible Peaks
    'dam': 'ad10c6d0-0aad-490f-a82c-6bf7ce67cd58', # ATAC: Differentially Accessible Motifs
    'dep': '38b59e4a-3c87-460d-b3e2-fd5ed62e7264', # ADT: Differentially Expressed Proteins
    'dgm': '06039bda-26a0-4a51-85b4-f2b5aedcdf74'  # RNA + ATAC: DEG Gene Motifs
}

In [6]:
in_files = {}
for k, uuid in in_uuids.items():
    in_files[k] = hisepy.cache_files([uuid])[0]

2026-06-15 21:26:57,348 INFO [hisepy.logging:175] logging 14664 139138472589120 Calling cache_files
2026-06-15 21:27:03,266 INFO [hisepy.logging:208] logging 14664 139138472589120 Finished cache_files, success=True, time_elapsed=3.952s
2026-06-15 21:27:03,267 INFO [hisepy.logging:175] logging 14664 139138472589120 Calling cache_files
2026-06-15 21:27:07,620 INFO [hisepy.logging:208] logging 14664 139138472589120 Finished cache_files, success=True, time_elapsed=2.862s
2026-06-15 21:27:07,621 INFO [hisepy.logging:175] logging 14664 139138472589120 Calling cache_files
2026-06-15 21:27:12,125 INFO [hisepy.logging:208] logging 14664 139138472589120 Finished cache_files, success=True, time_elapsed=2.966s
2026-06-15 21:27:12,125 INFO [hisepy.logging:175] logging 14664 139138472589120 Calling cache_files
2026-06-15 21:27:16,274 INFO [hisepy.logging:208] logging 14664 139138472589120 Finished cache_files, success=True, time_elapsed=2.756s
2026-06-15 21:27:16,274 INFO [hisepy.logging:175] loggin

In [7]:
cell_type_update = {
    "t_cd4_cm": "CD4 CM",
    "t_cd4_em": "CD4 EM",
    "t_cd4_naive": "CD4 Naive",
    "t_cd4_treg": "CD4 Treg",
    "t_cd8_memory": "CD8 Memory",
    "t_cd8_naive": "CD8 Naive"
}
treatment_update = {
    "bortezomib": "Bortezomib",
    "dexamethasone": "Dexamethasone",
    "dmso": "DMSO",
    "lenalidomide": "Lenalidomide",
    "untreated": "Untreated"
}
timepoint_update = {
    "0": "T0",
    "4": "T4",
    "24": "T24",
    "72": "T72"
}

## DEGs

In [8]:
deg = pl.read_csv(in_files['deg'], infer_schema_length=100000)

In [9]:
deg.head()

aifi_cell_type,timepoint,fg,bg,n_sample,gene,coef_C,coef_D,logFC,nomP,adjP,mean
str,i64,str,str,i64,str,str,f64,str,f64,f64,f64
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""A1BG-AS1""","""0.0284117728331221""",-0.127145,"""-0.00860394028790527""",0.74037,0.999237,0.085197
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""AAGAB""","""0.0322872380440183""",-0.25575,"""-0.05429508746593""",0.145599,0.991887,0.29036
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""AAK1""","""-0.0090581491256185""",0.047696,"""0.0148158042264956""",0.876254,0.999237,0.895006
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""AAMDC""","""0.0622948443487934""",0.192507,"""0.0431354534815564""",0.160231,0.991887,0.2063
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""AAMP""","""0.0180806843790714""",-0.166733,"""-0.0100120280827752""",0.733253,0.999237,0.068101


In [10]:
deg = deg.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in deg['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = [timepoint_update[x] for x in deg['timepoint'].cast(pl.String)]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[x] for x in deg['fg']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[x] for x in deg['fg']]
    ),
    pl.Series(
        name = 'bg',
        values = [treatment_update[x] for x in deg['bg']]
    ),
    pl.col('logFC').replace("NA", None).cast(pl.Float32),
    pl.col('nomP').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32)
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'gene',
     'n_sample', 'mean', 'logFC', 'nomP', 'adjP']
)

In [11]:
deg.head()

Cell Type,Treatment,Timepoint,fg,bg,gene,n_sample,mean,logFC,nomP,adjP
str,str,str,str,str,str,i64,f64,f32,f32,f32
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""A1BG-AS1""",648,0.085197,-0.008604,0.74037,0.999237
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAGAB""",648,0.29036,-0.054295,0.145599,0.991887
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAK1""",648,0.895006,0.014816,0.876254,0.999237
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAMDC""",648,0.2063,0.043135,0.160231,0.991887
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""AAMP""",648,0.068101,-0.010012,0.733253,0.999237


In [12]:
out_deg = 'output/tcell-vrd_deg_{d}.csv'.format(d = date.today())
deg.write_csv(out_deg)
out_files.append(out_deg)

## Hallmark GSEA

In [13]:
hallmark = pl.read_csv(in_files['hallmark'], infer_schema_length=100000)

In [14]:
hallmark.head()

fg,bg,timepoint,aifi_cell_type,pathway_label,NES,nomP,adjP,n_leadingEdge,n_pathway_genes,leadingEdge,pathway_genes
str,str,i64,str,str,f64,f64,f64,i64,i64,str,str
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""MYC Targets V1""",2.01374,7.2040e-8,0.000003,180,236,"""PSMD1;PSMD14;PSMB2;PSMD7;PSMA6…","""ABCE1;ACP1;AIMP2;AP3S1;APEX1;B…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""MTORC1 Signaling""",1.817862,0.000223,0.003499,142,211,"""PSMD14;TXNRD1;PSMD12;PSMC6;SQS…","""ABCF2;ACACA;ACACA;ACLY;ACSL3;A…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""ROS Pathway""",1.755945,0.009262,0.054414,38,58,"""TXNRD1;TXN;GSR;FTL;PRDX1;GCLM;…","""ABCC1;ABCC1;ATOX1;CAT;CDKN2D;E…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""Adipogenesis""",1.678625,0.003415,0.026212,121,210,"""UBC;UBQLN1;TALDO1;NMT1;BAZ2A;S…","""ABCA1;ABCB8;ACAA2;ACADL;ACADM;…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""Xenobiotic Metabolism""",1.396928,0.093089,0.266258,73,224,"""GABARAPL1;NMT1;GSR;PTGES3;ACO2…","""ABCC2;ABCC3;ABCD2;ABHD6;ACO2;A…"


In [15]:
hallmark = hallmark.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in hallmark['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = [timepoint_update[x] for x in hallmark['timepoint'].cast(pl.String)]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[x] for x in hallmark['fg']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[x] for x in hallmark['fg']]
    ),
    pl.Series(
        name = 'bg',
        values = [treatment_update[x] for x in hallmark['bg']]
    ),
    pl.col('NES').cast(pl.Float32),
    pl.col('nomP').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32)
).rename(
    {'pathway_label': 'pathway',
     'n_leadingEdge': 'n_leading_edge_genes',
     'leadingEdge': 'leading_edge_genes'}
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'pathway',
     'NES', 'nomP', 'adjP', 'n_pathway_genes', 'n_leading_edge_genes', 'leading_edge_genes']
)

In [16]:
hallmark.head()

Cell Type,Treatment,Timepoint,fg,bg,pathway,NES,nomP,adjP,n_pathway_genes,n_leading_edge_genes,leading_edge_genes
str,str,str,str,str,str,f32,f32,f32,i64,i64,str
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""MYC Targets V1""",2.01374,7.2040e-8,0.000003,236,180,"""PSMD1;PSMD14;PSMB2;PSMD7;PSMA6…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""MTORC1 Signaling""",1.817862,0.000223,0.003499,211,142,"""PSMD14;TXNRD1;PSMD12;PSMC6;SQS…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ROS Pathway""",1.755945,0.009262,0.054414,58,38,"""TXNRD1;TXN;GSR;FTL;PRDX1;GCLM;…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""Adipogenesis""",1.678625,0.003415,0.026212,210,121,"""UBC;UBQLN1;TALDO1;NMT1;BAZ2A;S…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""Xenobiotic Metabolism""",1.396928,0.093089,0.266258,224,73,"""GABARAPL1;NMT1;GSR;PTGES3;ACO2…"


In [17]:
out_hallmark = 'output/tcell-vrd_gsea_hallmark_{d}.tsv'.format(d = date.today())
hallmark.write_csv(out_hallmark, separator = '\t')
out_files.append(out_hallmark)

## Reactome GSEA

In [18]:
reactome = pl.read_csv(in_files['reactome'], infer_schema_length=100000, separator = '\t')

In [19]:
reactome.head()

fg,bg,timepoint,aifi_cell_type,id,pathway,NES,nomP,adjP,n_leadingEdge,n_pathway_genes,leadingEdge,pathway_genes
str,str,i64,str,str,str,f64,f64,f64,i64,i64,str,str
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""R-HSA-8951664""","""Neddylation""",2.37075,5.8764e-22,5.8647e-19,168,247,"""PSMD1;PSMD14;VCP;UBC;PSMD11;NP…","""AMER1;ANKRD9;ASB1;ASB10;ASB11;…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""R-HSA-5619115""","""Disorders of transmembrane tra…",2.34676,1.8173e-19,9.0684e-17,106,177,"""PSMD1;PSMD14;VCP;UBC;PSMD11;PS…","""AAAS;ABCA1;ABCA12;ABCA3;ABCB11…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""R-HSA-9755511""","""KEAP1-NFE2L2 pathway""",2.344083,8.2746e-18,9.1756e-16,94,132,"""PSMD1;PSMD14;VCP;UBC;PSMD11;NP…","""ABCC1;ABCC3;ABCF2;ABCG2;AKT1;A…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""R-HSA-453274""","""Mitotic G2-G2/M phases""",2.342449,3.7972e-18,6.2888e-16,142,199,"""PSMD1;PSMD14;UBC;PSMD11;PSMB2;…","""ACTR1A;AJUBA;AKAP9;ALMS1;AURKA…"
"""bortezomib""","""dmso""",24,"""t_cd4_cm""","""R-HSA-69275""","""G2/M Transition""",2.33471,1.8126e-17,1.5075e-15,141,197,"""PSMD1;PSMD14;UBC;PSMD11;PSMB2;…","""ACTR1A;AJUBA;AKAP9;ALMS1;AURKA…"


In [20]:
reactome = reactome.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in reactome['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = [timepoint_update[x] for x in reactome['timepoint'].cast(pl.String)]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[x] for x in reactome['fg']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[x] for x in reactome['fg']]
    ),
    pl.Series(
        name = 'bg',
        values = [treatment_update[x] for x in reactome['bg']]
    ),
    pl.col('NES').cast(pl.Float32),
    pl.col('nomP').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32)
).rename(
    {'n_leadingEdge': 'n_leading_edge_genes',
     'leadingEdge': 'leading_edge_genes'}
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'pathway',
     'NES', 'nomP', 'adjP', 'n_pathway_genes', 'n_leading_edge_genes', 'leading_edge_genes']
)

In [21]:
reactome.head()

Cell Type,Treatment,Timepoint,fg,bg,pathway,NES,nomP,adjP,n_pathway_genes,n_leading_edge_genes,leading_edge_genes
str,str,str,str,str,str,f32,f32,f32,i64,i64,str
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""Neddylation""",2.37075,5.8764e-22,5.8647e-19,247,168,"""PSMD1;PSMD14;VCP;UBC;PSMD11;NP…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""Disorders of transmembrane tra…",2.34676,1.8173e-19,9.0684e-17,177,106,"""PSMD1;PSMD14;VCP;UBC;PSMD11;PS…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""KEAP1-NFE2L2 pathway""",2.344083,8.2746e-18,9.1756e-16,132,94,"""PSMD1;PSMD14;VCP;UBC;PSMD11;NP…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""Mitotic G2-G2/M phases""",2.342449,3.7972e-18,6.2888e-16,199,142,"""PSMD1;PSMD14;UBC;PSMD11;PSMB2;…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""G2/M Transition""",2.33471,1.8126e-17,1.5075e-15,197,141,"""PSMD1;PSMD14;UBC;PSMD11;PSMB2;…"


In [22]:
out_reactome = 'output/tcell-vrd_gsea_reactome_{d}.tsv'.format(d = date.today())
hallmark.write_csv(out_reactome, separator = '\t')
out_files.append(out_reactome)

## DAP

In [23]:
dap = pl.read_csv(in_files['dap'], infer_schema_length=100000)

In [24]:
dap.head()

aifi_cell_type,fg,bg,seqnames,start,end,logFC,adjP,MeanDiff,idx
str,str,str,str,i64,i64,f64,f64,f64,i64
"""t_cd4_cm""","""bortezomib_4""","""dmso_4""","""chr1""",817097,817597,0.762472,0.961701,0.016549,1
"""t_cd4_cm""","""bortezomib_4""","""dmso_4""","""chr1""",827316,827816,-0.012146,0.961701,-0.002062,2
"""t_cd4_cm""","""bortezomib_4""","""dmso_4""","""chr1""",844402,844902,-0.720519,0.961701,-0.017751,3
"""t_cd4_cm""","""bortezomib_4""","""dmso_4""","""chr1""",903980,904480,-1.176057,0.961701,-0.016888,6
"""t_cd4_cm""","""bortezomib_4""","""dmso_4""","""chr1""",904495,904995,-0.215636,0.961701,-0.040113,7


In [25]:
dap = dap.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in dap['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = ['T' + re.sub('.+_','',x) for x in dap['fg']]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[re.sub('_.+','',x)] for x in dap['fg']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[re.sub('_.+','',x)] for x in dap['fg']]
    ),
    pl.Series(
        name = 'bg',
        values = [treatment_update[re.sub('_.+','',x)] for x in dap['bg']]
    ),
    pl.col('MeanDiff').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32),
    pl.concat_str(
        [pl.col('seqnames'),pl.lit(':'),pl.col('start'),pl.lit('-'),pl.col('end')]
    ).alias('peak')
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'peak',
     'MeanDiff', 'adjP']
)

In [26]:
dap.head()

Cell Type,Treatment,Timepoint,fg,bg,peak,MeanDiff,adjP
str,str,str,str,str,str,f32,f32
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""chr1:817097-817597""",0.016549,0.961701
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""chr1:827316-827816""",-0.002062,0.961701
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""chr1:844402-844902""",-0.017751,0.961701
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""chr1:903980-904480""",-0.016888,0.961701
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""chr1:904495-904995""",-0.040113,0.961701


In [27]:
out_dap = 'output/tcell-vrd_dap_{d}.csv'.format(d = date.today())
dap.write_csv(out_dap)
out_files.append(out_dap)

## DAM

In [28]:
dam = pl.read_csv(in_files['dam'], infer_schema_length=100000)

In [29]:
dam.tail()

aifi_cell_type,fg,bg,direction,feature,tf_gene,Enrichment,nomP,adjP,mlog10Padj,mlog10p,BackgroundProporition,nBackground,BackgroundFrequency,CompareProportion,nCompare,CompareFrequency
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,i64,i64,str,i64,i64
"""t_cd8_naive""","""lenalidomide_72""","""dmso_72""","""dn""","""SMAD5_866""","""SMAD5""","""1.30615631349722""",0.007018,0.06316,1.199557,2.1538,0.125811,73070,9193,"""0.164328657314629""",499,82
"""t_cd8_naive""","""lenalidomide_72""","""dmso_72""","""dn""","""SMAD9_867""","""SMAD9""","""0.929505495365731""",0.67733,1.0,0.0,0.1692,0.056056,73070,4096,"""0.0521042084168337""",499,26
"""t_cd8_naive""","""lenalidomide_72""","""dmso_72""","""dn""","""SOX6_868""","""SOX6""","""0.94209435383742""",0.658719,1.0,0.0,0.1813,0.063816,73070,4663,"""0.0601202404809619""",499,30
"""t_cd8_naive""","""lenalidomide_72""","""dmso_72""","""dn""","""TBX18_869""","""TBX18""","""0.754079772031994""",0.903025,1.0,0.0,0.0443,0.042521,73070,3107,"""0.032064128256513""",499,16
"""t_cd8_naive""","""lenalidomide_72""","""dmso_72""","""dn""","""TBX22_870""","""TBX22""","""0.754079772031994""",0.903025,1.0,0.0,0.0443,0.042521,73070,3107,"""0.032064128256513""",499,16


In [30]:
dam = dam.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in dam['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = ['T' + re.sub('.+_','',x) for x in dam['fg']]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[re.sub('_.+','',x)] for x in dam['fg']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[re.sub('_.+','',x)] for x in dam['fg']]
    ),
    pl.Series(
        name = 'bg',
        values = [treatment_update[re.sub('_.+','',x)] for x in dam['bg']]
    ),
    pl.col('Enrichment').replace('NA', None).cast(pl.Float32),
    pl.col('CompareProportion').replace('NA', None).cast(pl.Float32),
    pl.col('nomP').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32)
).rename(
    {'feature': 'motif_id',
     'BackgroundProporition': 'bg_prop',
     'nBackground': 'bg_n',
     'BackgroundFrequency': 'bg_freq',
     'CompareProportion': 'fg_prop',
     'nCompare': 'fg_n',
     'CompareFrequency': 'fg_freq'
    }
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'motif_id', 'tf_gene',
     'Enrichment', 'nomP', 'adjP',
     'bg_n', 'bg_freq', 'bg_prop', 'fg_n', 'fg_freq', 'fg_prop']
)

In [31]:
dam.head()

Cell Type,Treatment,Timepoint,fg,bg,motif_id,tf_gene,Enrichment,nomP,adjP,bg_n,bg_freq,bg_prop,fg_n,fg_freq,fg_prop
str,str,str,str,str,str,str,f32,f32,f32,i64,i64,f64,i64,i64,f32
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""TFAP2B_1""","""TFAP2B""",null,1.0,1.0,72737,4986,0.068548,0,0,null
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""TFAP2D_2""","""TFAP2D""",null,1.0,1.0,72737,12274,0.168745,0,0,null
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""TFAP2C_3""","""TFAP2C""",null,1.0,1.0,72737,10565,0.145249,0,0,null
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""TFAP2E_4""","""TFAP2E""",null,1.0,1.0,72737,3579,0.049205,0,0,null
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""TFAP2A_5""","""TFAP2A""",null,1.0,1.0,72737,5968,0.082049,0,0,null


In [32]:
out_dam = 'output/tcell-vrd_dam_{d}.csv'.format(d = date.today())
dam.write_csv(out_dam)
out_files.append(out_dam)

## DEP

In [33]:
dep = pl.read_csv(in_files['dep'], infer_schema_length=100000)

In [34]:
dep.head()

aifi_cell_type,timepoint,fg,bg,n_downsample,feature,estimate,std_error,t_value,nomP,mean,fg_mean,bg_mean,fc,adjP,logFC
str,i64,str,str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""CD11c""",0.005166,0.0067,0.771099,0.440789,0.124462,0.127046,0.121879,1.04239,0.795736,0.059894
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""CD278""",-0.001688,0.015431,-0.10942,0.912887,0.51246,0.511616,0.513305,0.996711,0.983653,-0.004753
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""CD11b""",-0.004868,0.00344,-1.415075,0.157287,0.03344,0.031006,0.035874,0.864311,0.507491,-0.210377
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""CD16""",0.005356,0.00576,0.929872,0.352611,0.109818,0.112496,0.10714,1.049994,0.743134,0.070382
"""t_cd4_cm""",4,"""bortezomib""","""dmso""",648,"""CD21""",0.00911,0.008902,1.023314,0.306351,0.289668,0.294223,0.285113,1.031952,0.694816,0.045376


In [36]:
dep = dep.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in dep['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = [timepoint_update[x] for x in dep['timepoint'].cast(pl.String)]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[x] for x in dep['fg']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[x] for x in dep['fg']]
    ),
    pl.Series(
        name = 'bg',
        values = [treatment_update[x] for x in dep['bg']]
    ),
    pl.col('logFC').cast(pl.Float32),
    pl.col('nomP').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32)
).rename(
    {'n_downsample': 'n_sample'}
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'feature',
     'n_sample', 'mean', 'logFC', 'nomP', 'adjP']
)

In [37]:
dep.head()

Cell Type,Treatment,Timepoint,fg,bg,feature,n_sample,mean,logFC,nomP,adjP
str,str,str,str,str,str,i64,f64,f32,f32,f32
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""CD11c""",648,0.124462,0.059894,0.440789,0.795736
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""CD278""",648,0.51246,-0.004753,0.912887,0.983653
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""CD11b""",648,0.03344,-0.210377,0.157287,0.507491
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""CD16""",648,0.109818,0.070382,0.352611,0.743133
"""CD4 CM""","""Bortezomib""","""T4""","""Bortezomib""","""DMSO""","""CD21""",648,0.289668,0.045376,0.306351,0.694816


In [38]:
out_dep = 'output/tcell-vrd_dep_{d}.csv'.format(d = date.today())
dep.write_csv(out_dep)
out_files.append(out_dep)

## DGM

Format as for GSEA enrichment for use in data app visualization

In [39]:
dgm = pl.read_csv(in_files['dgm'], infer_schema_length=100000)

In [40]:
dgm['direction'].value_counts()

direction,count
str,u32
"""all""",10840
"""up""",10840
"""dn""",10840


In [41]:
dgm.head()

treatment,timepoint,aifi_cell_type,direction,motif_id,tf_gene,tf_logFC,tf_adjP,n_all_genes,n_motif_genes,n_deg,n_ol,nomP,ol_genes,adjP
str,i64,str,str,str,str,str,f64,i64,i64,i64,i64,f64,str,f64
"""bortezomib""",24,"""t_cd4_cm""","""all""","""ARID3A_6""","""ARID3A""","""0.000931044552036991""",0.981924,7606,4691,528,341,0.083555,"""ABAT;ABHD4;ABI1;ABLIM1;ACAP2;A…",0.179853
"""bortezomib""",24,"""t_cd4_cm""","""all""","""ARID5B_7""","""ARID5B""","""-0.33967153610607""",4.8125e-7,7606,1304,528,104,0.061935,"""ABLIM1;ACVR1C;ADAM19;AHNAK;ALS…",0.138343
"""bortezomib""",24,"""t_cd4_cm""","""all""","""ARID3B_8""","""ARID3B""","""0.127660815204762""",0.007706,7606,2329,528,193,0.001451,"""ABAT;ABI1;ACO2;ACTB;ADAM19;ADH…",0.005219
"""bortezomib""",24,"""t_cd4_cm""","""all""","""ARID5A_9""","""ARID5A""","""-0.0240395489595338""",0.781995,7606,2020,528,190,4.9349e-7,"""ACAP2;ADAM19;AHNAK;AIM2;ALS2;A…",0.000005
"""bortezomib""",24,"""t_cd4_cm""","""all""","""ARID2_11""","""ARID2""","""-0.0698178004795795""",0.633254,7606,943,528,96,0.000045,"""ABHD3;ACAP2;ADAM19;ADGRE5;AGTP…",0.000253


In [42]:
dgm = dgm.with_columns(
    pl.Series(
        name = 'Cell Type',
        values = [cell_type_update[x] for x in dgm['aifi_cell_type']]
    ),
    pl.Series(
        name = 'Timepoint',
        values = [timepoint_update[x] for x in dgm['timepoint'].cast(pl.String)]
    ),
    pl.Series(
        name = 'Treatment',
        values = [treatment_update[x] for x in dgm['treatment']]
    ),
    pl.Series(
        name = 'fg',
        values = [treatment_update[x] for x in dgm['treatment']]
    ),
    pl.lit('DMSO').alias('bg'),
    pl.col('tf_logFC').replace('NA', None).cast(pl.Float32),
    pl.col('nomP').cast(pl.Float32),
    pl.col('adjP').cast(pl.Float32)
).rename(
    {'n_all_genes': 'n_genes',
     'n_all_genes': 'n_tested_genes',
     'n_motif_genes': 'n_tested_genes_near_motif',
     'n_ol': 'n_deg_near_motif',
     'ol_genes': 'deg_near_motif',
     'direction': 'deg_direction'}
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'deg_direction',
     'motif_id', 'tf_gene', 'tf_logFC', 'tf_adjP',
     'n_tested_genes', 'n_tested_genes_near_motif', 'n_deg', 'n_deg_near_motif',
     'nomP', 'adjP', 'deg_near_motif']
)

In [43]:
dgm.head()

Cell Type,Treatment,Timepoint,fg,bg,deg_direction,motif_id,tf_gene,tf_logFC,tf_adjP,n_tested_genes,n_tested_genes_near_motif,n_deg,n_deg_near_motif,nomP,adjP,deg_near_motif
str,str,str,str,str,str,str,str,f32,f64,i64,i64,i64,i64,f32,f32,str
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""all""","""ARID3A_6""","""ARID3A""",0.000931,0.981924,7606,4691,528,341,0.083555,0.179853,"""ABAT;ABHD4;ABI1;ABLIM1;ACAP2;A…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""all""","""ARID5B_7""","""ARID5B""",-0.339672,4.8125e-7,7606,1304,528,104,0.061935,0.138343,"""ABLIM1;ACVR1C;ADAM19;AHNAK;ALS…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""all""","""ARID3B_8""","""ARID3B""",0.127661,0.007706,7606,2329,528,193,0.001451,0.005219,"""ABAT;ABI1;ACO2;ACTB;ADAM19;ADH…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""all""","""ARID5A_9""","""ARID5A""",-0.02404,0.781995,7606,2020,528,190,4.9349e-7,0.000005,"""ACAP2;ADAM19;AHNAK;AIM2;ALS2;A…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""all""","""ARID2_11""","""ARID2""",-0.069818,0.633254,7606,943,528,96,0.000045,0.000253,"""ABHD3;ACAP2;ADAM19;ADGRE5;AGTP…"


In [44]:
out_dgm = 'output/tcell-vrd_dgm_{d}.csv'.format(d = date.today())
dgm.write_csv(out_dgm)
out_files.append(out_dgm)

To treat DGM like pathways, we'll compute an enrichment based on the background expectation of `frac_expected = n_tested_genes_near_motif / n_tested_genes`. We'll then use `frac_deg = n_deg_near_motif / n_deg` as the enriched proportion. `frac_deg / frac_expected` will be our enrichment score, which we'll add to the "NES" column for consistency with pathway results.

We'll also use just the up- and down-regulated deg sets, and assign a negative score to the down-regulated DEG sets so that they align with GSEA analyses.

In [45]:
dgm_pathway_structure = dgm.filter(
    pl.col('deg_direction').is_in(['up','dn'])
).with_columns(
    (pl.col('n_tested_genes_near_motif') / pl.col('n_tested_genes')).alias('frac_expected'),
    (pl.col('n_deg_near_motif') / pl.col('n_deg')).alias('frac_deg')
).with_columns(
    pl.when(
        pl.col('deg_direction') == 'up'
    ).then(
        pl.col('frac_deg') / pl.col('frac_expected')
    ).otherwise(
        -1 * pl.col('frac_deg') / pl.col('frac_expected')
    ).alias('NES')
).rename(
    {'motif_id': 'pathway',
     'n_deg': 'n_pathway_genes',
     'n_deg_near_motif': 'n_leading_edge_genes',
     'deg_near_motif': 'leading_edge_genes'
    }
).select(
    ['Cell Type', 'Treatment', 'Timepoint', 'fg', 'bg', 'pathway',
     'NES', 'nomP', 'adjP', 'n_pathway_genes', 'n_leading_edge_genes', 'leading_edge_genes']
)

In [46]:
dgm_pathway_structure.filter(pl.col('adjP') < 0.05).filter(pl.col('NES') < 0).head(20)

Cell Type,Treatment,Timepoint,fg,bg,pathway,NES,nomP,adjP,n_pathway_genes,n_leading_edge_genes,leading_edge_genes
str,str,str,str,str,str,f64,f32,f32,i64,i64,str
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ARID3A_6""",-1.175517,0.000738,0.002904,200,145,"""ABLIM1;ACTB;ACVR1C;ADAM19;ADGR…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ARID5B_7""",-1.749847,0.000004,0.00003,200,60,"""ABLIM1;ACVR1C;ADAM19;AHNAK;ANK…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ARID3B_8""",-1.600232,2.7856e-8,4.0011e-7,200,98,"""ACTB;ADAM19;ANK3;ANTXR2;ARHGAP…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ARID5A_9""",-1.84502,5.4146e-12,2.0831e-10,200,98,"""ADAM19;AHNAK;ANK3;ANTXR2;ARHGA…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ARID2_11""",-2.218081,4.3224e-9,7.8048e-8,200,55,"""ADAM19;ADGRE5;ANK3;ANTXR2;ARL4…"
…,…,…,…,…,…,…,…,…,…,…,…
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ID3_38""",-1.617255,1.1887e-8,1.9108e-7,200,99,"""ACTB;AHNAK;ANK3;ANTXR2;ARHGAP1…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""MXI1_40""",-2.255772,1.3116e-13,7.2664e-12,200,78,"""ACTB;ADAM19;ADGRE5;AHNAK;ANK3;…"
"""CD4 CM""","""Bortezomib""","""T24""","""Bortezomib""","""DMSO""","""ARNTL_50""",-2.054542,4.3964e-7,0.000005,200,49,"""ADGRE5;AHNAK;ANTXR2;ARHGAP15;A…"


In [47]:
out_gsea_dgm = 'output/tcell-vrd_gsea_dgm_{d}.csv'.format(d = date.today())
dgm_pathway_structure.write_csv(out_gsea_dgm)
out_files.append(out_gsea_dgm)

## Upload results to HISE

In [48]:
study_space_uuid = '40df6403-29f0-4b45-ab7d-f46d420c422e'
title = 'VRd TEA-seq formatted differential tables {d}'.format(d = date.today())

In [49]:
search_id = element_id()
search_id

'gold-neptunium-aluminum'

In [50]:
in_files = list(in_uuids.values())

In [51]:
in_files

['a134530f-9ac4-4d37-bf01-97c73bf1ce24',
 '4f05f2a0-59d2-4d04-b948-70d541613e71',
 'ad8d7320-399c-41ff-8b2f-b840d8eae324',
 '681e1f11-708c-4ab4-8ed0-87ef895df152',
 'ad10c6d0-0aad-490f-a82c-6bf7ce67cd58',
 '38b59e4a-3c87-460d-b3e2-fd5ed62e7264',
 '06039bda-26a0-4a51-85b4-f2b5aedcdf74']

In [52]:
out_files

['output/tcell-vrd_deg_2026-06-15.csv',
 'output/tcell-vrd_gsea_hallmark_2026-06-15.tsv',
 'output/tcell-vrd_gsea_reactome_2026-06-15.tsv',
 'output/tcell-vrd_dap_2026-06-15.csv',
 'output/tcell-vrd_dam_2026-06-15.csv',
 'output/tcell-vrd_dep_2026-06-15.csv',
 'output/tcell-vrd_dgm_2026-06-15.csv',
 'output/tcell-vrd_gsea_dgm_2026-06-15.csv']

In [53]:
import session_info
session_info.show()

In [54]:
hisepy.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

2026-06-15 21:28:28,960 INFO [hisepy.logging:175] logging 14664 139138472589120 Calling upload_files


Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


2026-06-15 21:28:32,433 INFO [hisepy.logging:175] logging 14664 139138472589120 Calling get_default_store
2026-06-15 21:28:34,201 INFO [hisepy.logging:208] logging 14664 139138472589120 Finished get_default_store, success=True, time_elapsed=0.403s
2026-06-15 21:28:48,616 INFO [hisepy.logging:208] logging 14664 139138472589120 Finished upload_files, success=True, time_elapsed=18.208s


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '27b4720f-0346-427d-a951-5b2ebb826248',
 'ProcessId': '3adf67dd-552e-4de9-a800-ce247fc9e28a',
 'WorkflowId': '5ae3401e-9581-4f32-a6c9-b6caf04d3e98',
 'FileIds': ['3279167f-a71c-442b-a325-2f778b4b07a1',
  '72224905-7a0e-439b-ae55-b582b72a2a69',
  'c33f14e3-f884-4272-b6c9-cf3f9b3a05c0',
  'cb748e0a-076f-46dc-9a9e-d24541f2c3ef',
  '6565ea13-2b82-40bb-b150-e42324c7144a',
  'c8c3b24b-286d-462d-ad88-206d78fc6b7f',
  '8276b63f-2c20-47ca-8dc7-f9b6285933b4',
  '5b1726a5-a49e-4882-a8a7-caa7c15fcfa1']}